In [ ]:
"""
cnn_model_single.py — Entrenamiento de CNN para UNA combinación de ventanas
=============================================================================

Cambia los parámetros en la sección CONFIGURACIÓN y ejecuta.
"""
import os
import sys
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Apuntar a la raíz del repositorio
os.chdir(r'C:\Users\eneko\neural-network-forecasting')
sys.path.insert(0, os.getcwd())

# Imports del repositorio
from config import RANDOM_SEED, MODELS_DIR, FIGURES_DIR
from src.data_pipeline import load_data, get_train_test
from src.evaluation import compute_mae, save_results, count_parameters
from src.plotting import plot_training_curves

np.random.seed(RANDOM_SEED)
import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from src.data_pipeline import load_data_lopez_de_prado, create_time_series_data
from sklearn.model_selection import train_test_split
from config import TEST_SIZE

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
from tensorflow.keras.layers import SpatialDropout1D

# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 10          
OUTPUT_WINDOW = 30        
FILTERS = 16              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2             
FILTERS_2 = 32            
KERNEL_SIZE_2 = 3         
FILTERS_3 = 0             
KERNEL_SIZE_3 = 3
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0         
USE_GAP = True
USE_MAXPOOL = False            
DENSE_1 = 32              
DENSE_2 = 16             
DENSE_3 = 0              
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.001
EPOCHS = 1000
BATCH_SIZE = 64
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,BatchNormalization(),
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

SyntaxError: positional argument follows keyword argument (1656758050.py, line 80)

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 10          
OUTPUT_WINDOW = 90        
FILTERS = 64              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 0             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = True            
DENSE_1 = 128              
DENSE_2 = 64            
DENSE_3 = 16             
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.00001
EPOCHS = 1000
BATCH_SIZE = 132
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 10          
OUTPUT_WINDOW = 5        
FILTERS = 32              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 64            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 0             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = False            
DENSE_1 = 64              
DENSE_2 = 8            
DENSE_3 = 0             
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.00001
EPOCHS = 1000
BATCH_SIZE = 32
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 10          
OUTPUT_WINDOW = 1        
FILTERS = 64              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 0             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = False            
DENSE_1 = 64              
DENSE_2 = 32            
DENSE_3 = 0             
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.000           
LEARNING_RATE = 0.0000001
EPOCHS = 1000
BATCH_SIZE = 256
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 5          
OUTPUT_WINDOW = 1        
FILTERS = 32              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 64            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 0             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = False            
DENSE_1 = 32              
DENSE_2 = 16            
DENSE_3 = 8            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.000           
LEARNING_RATE = 0.00001
EPOCHS = 1000
BATCH_SIZE = 256
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 5          
OUTPUT_WINDOW = 5        
FILTERS = 256              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 512            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 0             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = False            
DENSE_1 = 64              
DENSE_2 = 64            
DENSE_3 = 32            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.000001
EPOCHS = 1000
BATCH_SIZE = 64
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 5          
OUTPUT_WINDOW = 30        
FILTERS = 64              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 0             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = False            
DENSE_1 = 64              
DENSE_2 = 32            
DENSE_3 = 0            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.00001
EPOCHS = 1000
BATCH_SIZE = 64
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 5          
OUTPUT_WINDOW = 90        
FILTERS = 64              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 0             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = False            
DENSE_1 = 128              
DENSE_2 = 64            
DENSE_3 = 0            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.000001
EPOCHS = 1000
BATCH_SIZE = 64
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 90          
OUTPUT_WINDOW = 1        
FILTERS = 64              
KERNEL_SIZE = 5           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 5
##################         
FILTERS_3 = 256             
KERNEL_SIZE_3 = 5
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = True            
DENSE_1 = 64              
DENSE_2 = 32            
DENSE_3 = 8            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.000001
EPOCHS = 1000
BATCH_SIZE = 64
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 90          
OUTPUT_WINDOW = 5        
FILTERS = 32              
KERNEL_SIZE = 7           
PADDING = 'same'          
POOL_SIZE = 4
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 7
##################         
FILTERS_3 = 256             
KERNEL_SIZE_3 = 7
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = True            
DENSE_1 = 32              
DENSE_2 = 0            
DENSE_3 = 0            
DROPOUT_RATE_1 = 0.3      
DROPOUT_RATE_2 = 0.4      
DROPOUT_RATE_3 = 0.5      
L2_REG = 0.0001           
LEARNING_RATE = 0.000001
EPOCHS = 1000
BATCH_SIZE = 64
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 90          
OUTPUT_WINDOW = 30        
FILTERS = 128              
KERNEL_SIZE = 7           
PADDING = 'same'          
POOL_SIZE = 4
##################             
FILTERS_2 = 256            
KERNEL_SIZE_2 = 7
##################         
FILTERS_3 = 512             
KERNEL_SIZE_3 = 7
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = True            
DENSE_1 = 128              
DENSE_2 = 64            
DENSE_3 = 32            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.00001
EPOCHS = 1000
BATCH_SIZE = 128
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 90          
OUTPUT_WINDOW = 90        
FILTERS = 128              
KERNEL_SIZE = 7           
PADDING = 'same'          
POOL_SIZE = 4
##################             
FILTERS_2 = 256            
KERNEL_SIZE_2 = 7
##################         
FILTERS_3 = 512             
KERNEL_SIZE_3 = 7
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = True            
DENSE_1 = 256              
DENSE_2 = 128            
DENSE_3 = 32            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.000001
EPOCHS = 1000
BATCH_SIZE = 128
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 30          
OUTPUT_WINDOW = 1        
FILTERS = 64              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 256             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = True            
DENSE_1 = 128              
DENSE_2 = 64            
DENSE_3 = 0            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.0001
EPOCHS = 1000
BATCH_SIZE = 128
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 30          
OUTPUT_WINDOW = 5        
FILTERS = 64              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 256             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = True            
DENSE_1 = 256              
DENSE_2 = 128            
DENSE_3 = 64            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.00001
EPOCHS = 1000
BATCH_SIZE = 128
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 30          
OUTPUT_WINDOW = 30        
FILTERS = 64              
KERNEL_SIZE = 3           
PADDING = 'same'          
POOL_SIZE = 2
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 3
##################         
FILTERS_3 = 256             
KERNEL_SIZE_3 = 3
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = True            
DENSE_1 = 256              
DENSE_2 = 128            
DENSE_3 = 64            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.00001
EPOCHS = 1000
BATCH_SIZE = 128
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

In [ ]:
# ============================================================================
# ▶▶▶ CONFIGURACIÓN ◀◀◀
# ============================================================================

INPUT_WINDOW = 30          
OUTPUT_WINDOW = 90        
FILTERS = 64              
KERNEL_SIZE = 7           
PADDING = 'same'          
POOL_SIZE = 4
##################             
FILTERS_2 = 128            
KERNEL_SIZE_2 = 7
##################         
FILTERS_3 = 256             
KERNEL_SIZE_3 = 7
#################
FILTERS_4 = 0           
KERNEL_SIZE_4 = 0
#################         
USE_GAP = True
USE_MAXPOOL = True            
DENSE_1 = 256              
DENSE_2 = 128            
DENSE_3 = 64            
DROPOUT_RATE_1 = 0.0      
DROPOUT_RATE_2 = 0.0      
DROPOUT_RATE_3 = 0.0      
L2_REG = 0.0001           
LEARNING_RATE = 0.00001
EPOCHS = 1000
BATCH_SIZE = 128
SPATIAL_DROPOUT = 0.0
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_LP_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]
    if SPATIAL_DROPOUT > 0:
        layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))
    
    if FILTERS_4 > 0:
        layers.append(Conv1D(FILTERS_4, kernel_size=KERNEL_SIZE_4, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))
        if SPATIAL_DROPOUT > 0:
            layers.append(SpatialDropout1D(SPATIAL_DROPOUT))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()
returns_lp = load_data_lopez_de_prado(d=0.4, thres=1e-4)

indice_comun = returns_lp.index
returns = returns.loc[indice_comun]

X, _ = create_time_series_data(returns_lp, INPUT_WINDOW, OUTPUT_WINDOW)
_, y = create_time_series_data(returns, INPUT_WINDOW, OUTPUT_WINDOW)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False, random_state=RANDOM_SEED

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")